# 01 · SOG：schema 与自动建图

这一本确立**分体建图（SOG）的 schema**，并从 FinMCP-Bench 的调用日志里**自动建出一张 SOG**，对应论文 §3。

1. Schema：对象图（实体、属性、关系、规则）+ 操作图（操作的输入输出绑定到属性），两者只通过属性 id 相连；导出 JSON Schema。
2. 手工小例子：用 schema 写一张投顾场景的小图，看派生数据流、按对象范围披露、渲染接口桩。
3. 自动建图（论文 §3.4 第 1–5 步）：操作 → 属性对齐 → 实体归纳 → 关系归纳 → 派生与校验。
4. 第 6 步人工整理的示范。
5. 披露深度与接口桩大小。
6. 保存，供后面几本使用。

**这一本不调用任何外部接口，不花钱。**

In [1]:
%pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
browsergym-core 0.14.2 requires lxml<6.0.0,>=4.9, but you have lxml 6.1.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, time
import pandas as pd
from jif import config, finmcp, build, sog as S
from jif.sog import SOG, Entity, Property, Relation
pd.set_option("display.max_colwidth", 120)
print("数据目录：", config.DATA_DIR)

数据目录： /Users/nelson/Documents/JevAsIF/finmcp-analysis/data


## 1. Schema

`jif/sog.py` 用 Pydantic 定义 SOG，下面把它导出成 JSON Schema（`schema/sog.schema.json`），并列出每类对象的字段。

要点：
- 属性 id 统一写成 `实体.属性`，**两张图之间只通过属性 id 相连**。
- 操作的 `inputs[].property` / `outputs[].property` 指向属性；**数据流边不存储，由"谁产出、谁消费同一个属性"推导**。
- 输入参数带访问标注：`fill`（谁来填：agent / identity / host / const）、`check`（`traced` 表示必须取自前面的返回值）。
- `returns` 是返回值的形状树，用来渲染类型化接口桩和做字段检查；`entity` 标出这一层是哪个实体的记录，`map` 表示以代码为键的字典。

In [3]:
S.export_json_schema(config.SCHEMA_DIR / "sog.schema.json")
schema = json.loads((config.SCHEMA_DIR / "sog.schema.json").read_text(encoding="utf-8"))
print("顶层字段：", list(schema["properties"]))
for name in ["ObjectGraph", "Entity", "Property", "Relation", "OperationGraph", "Operation", "InputParam", "OutputBinding"]:
    d = schema["$defs"][name]
    print(f"\n{name}")
    for k, v in d["properties"].items():
        print(f"  {k:<15} {v.get('description', '')}")

顶层字段： ['sog_version', 'id', 'label', 'description', 'objects', 'operations', 'meta']

ObjectGraph
  entry           入口实体；按对象范围披露从这里出发
  entities        实体
  relations       实体之间的关系

Entity
  id              实体 id，如 Fund
  label           中文名称
  description     业务含义；会写进 Jev 的判断依据
  identifier      标识属性的 id
  label_property  给标识配可读名称的属性 id
  properties      属性列表
  rules           业务规则；会写进接口桩和 Jev 的判断依据

Property
  id              Entity.name，全图唯一
  name            数据中出现的字段名
  description     含义、口径、单位
  type            string / integer / number / boolean / array / object
  aliases         表示同一个值的其他字段名或参数名

Relation
  id              关系 id
  source          起点实体；披露沿 source → target 方向展开
  target          终点实体
  label           关系含义，如 可咨询、包含、引用
  via             承载这条关系的属性 id

OperationGraph
  operations      操作；数据流边不存储，由属性的产出与消费推导
  control_edges   可选的业务流程边

Operation
  id              操作 id，也是 agent 代码里调用的函数名
  tool            绑定的工具名（MCP 工具）
  label           中文名称
  description     功能说明，写进

## 2. 手工小例子

一张投顾"组合详解"的小图：客户 → 组合 → 费率规则。`read_fees` 需要的 `chargeNumber` 由 `read_basic` 产出，这条依赖**没有手工画**，是推导出来的。

In [4]:
obj = lambda **f: {"t": "object", "f": f}
toy = SOG.model_validate({
    "id": "advisory-toy", "label": "组合详解（手工示例）",
    "objects": {"entry": "Customer", "entities": [
        {"id": "Customer", "label": "客户", "identifier": "Customer.customerId",
         "properties": [{"id": "Customer.customerId", "name": "customerId"}]},
        {"id": "Portfolio", "label": "投顾组合", "identifier": "Portfolio.combineCode", "label_property": "Portfolio.combineName",
         "description": "客户可咨询的组合", "rules": ["组合类型不能根据名称推断"],
         "properties": [{"id": "Portfolio.combineCode", "name": "combineCode", "description": "组合代码"},
                        {"id": "Portfolio.combineName", "name": "combineName", "description": "组合名称"},
                        {"id": "Portfolio.chargeNumber", "name": "chargeNumber", "description": "费用编号"}]},
        {"id": "FeeSchedule", "label": "费率规则", "identifier": "FeeSchedule.chargeNumber",
         "rules": ["标准费率不是客户实际扣费"],
         "properties": [{"id": "FeeSchedule.chargeNumber", "name": "chargeNumber"},
                        {"id": "FeeSchedule.fareRate", "name": "fareRate", "description": "服务费率原值"}]}],
        "relations": [{"id": "r1", "source": "Customer", "target": "Portfolio", "label": "可咨询"},
                      {"id": "r2", "source": "Portfolio", "target": "FeeSchedule", "label": "适用计费结构"}]},
    "operations": {"operations": [
        {"id": "read_scope", "tool": "strategy.read_scope", "description": "获取客户可咨询的组合",
         "inputs": [{"name": "customerId", "property": "Customer.customerId", "fill": "identity", "required": True}],
         "outputs": [{"path": "portfolios[*].combineCode", "property": "Portfolio.combineCode"},
                     {"path": "portfolios[*].combineName", "property": "Portfolio.combineName"}],
         "returns": obj(portfolios={"t": "array", "i": {**obj(combineCode={"t": "string"}, combineName={"t": "string"}), "entity": "Portfolio"}})},
        {"id": "read_basic", "tool": "strategy.read_basic", "description": "获取组合基本资料",
         "inputs": [{"name": "combineCode", "property": "Portfolio.combineCode", "required": True, "check": ["traced"]}],
         "outputs": [{"path": "chargeNumber", "property": "FeeSchedule.chargeNumber"}],
         "returns": {**obj(combineCode={"t": "string"}, chargeNumber={"t": "integer"}), "entity": "Portfolio"}},
        {"id": "read_fees", "tool": "strategy.read_fees", "description": "获取标准计费结构",
         "inputs": [{"name": "chargeNumber", "type": "integer", "property": "FeeSchedule.chargeNumber", "required": True, "check": ["traced"]}],
         "outputs": [{"path": "fareRate", "property": "FeeSchedule.fareRate"}],
         "returns": {**obj(chargeNumber={"t": "integer"}, fareRate={"t": "string"}), "entity": "FeeSchedule"}}]}})

print("校验：", S.validate(toy) or "通过")
print("派生数据流：")
for e in S.derived_dataflow(toy):
    print("  ", " → ".join(e[:2]), f"（属性 {e[2]}）")
for h in (0, 1, 2):
    d = S.disclose(toy, h)
    print(f"披露深度 {h}：实体 {d['entities']}，操作 {d['operations']}，范围外 {d['frontier']}")

校验： 通过
派生数据流：
   read_scope → read_basic （属性 Portfolio.combineCode）
   read_basic → read_fees （属性 FeeSchedule.chargeNumber）
披露深度 0：实体 ['Customer']，操作 []，范围外 ['Portfolio']
披露深度 1：实体 ['Customer', 'Portfolio']，操作 ['read_scope']，范围外 ['FeeSchedule']
披露深度 2：实体 ['Customer', 'FeeSchedule', 'Portfolio']，操作 ['read_scope', 'read_basic', 'read_fees']，范围外 []


In [5]:
print(S.render_stubs(toy))
print("------ 平铺形式（只有参数，没有返回类型和实体信息）------")
print(S.render_stubs(toy, mode="flat"))

# ---- 实体（对象图） ----
# Portfolio（投顾组合）：标识字段 combineCode，名称字段 combineName。客户可咨询的组合
#   规则：组合类型不能根据名称推断
# FeeSchedule（费率规则）：标识字段 chargeNumber，名称字段 -。
#   规则：标准费率不是客户实际扣费

# ---- 操作（操作图）：只能用关键字参数调用 ----
def read_scope() -> read_scope_R:
    """获取客户可咨询的组合（customerId 由运行时注入，不要传）"""
    class read_scope_R:
        portfolios: list[read_scope_R_portfolios]
    class read_scope_R_portfolios:  # Portfolio 记录
        combineCode: str
        combineName: str

def read_basic(combineCode: str) -> read_basic_R:
    """获取组合基本资料 参数 combineCode 必须取自前面工具的返回值或客户原话。"""
    class read_basic_R:  # Portfolio 记录
        combineCode: str
        chargeNumber: int

def read_fees(chargeNumber: int) -> read_fees_R:
    """获取标准计费结构 参数 chargeNumber 必须取自前面工具的返回值或客户原话。"""
    class read_fees_R:  # FeeSchedule 记录
        chargeNumber: int
        fareRate: str

------ 平铺形式（只有参数，没有返回类型和实体信息）------
def read_scope(customerId: Any) -> Any:
    """获取客户可咨询的组合"""

def read_basic(combineCode: Any) -> Any:
    """获取组合基本资料"""



## 3. 从 FinMCP 日志自动建图

FinMCP-Bench 没有提供工具的参数定义，也没有实体模型，正好用来检验自动建图：

| 步骤 | 做法 |
| --- | --- |
| 1 操作 | 每个工具一个操作；输入取自调用参数，返回形状取自日志里的返回结果；以代码为键的字典折叠成 map |
| 2 属性对齐 | 以"工具 + 路径"为粒度做并查集：轨迹里观测到的值流动、足够具体的同名字段、值的包含关系；名字明确属于不同实体的簇（基金 / 策略）**不许合并** |
| 3 实体归纳 | 每个标识簇一个实体，按名字投票命名，选名称字段作标签 |
| 4 关系归纳 | 嵌套记录、同一记录里的其他标识、用户可检索 / 拥有的实体 |
| 5 派生与校验 | 推导数据流边，检查可达性；**用轨迹里实际发生的值流动检验派生边的召回率** |

In [6]:
t0 = time.time()
auto, report = build.build_sog(finmcp.iter_tasks(("multi_tool", "multi_turn")))
print(f"用时 {time.time() - t0:.1f} 秒；校验问题：{report['problems'] or '无'}")
pd.DataFrame([{k: report[k] for k in ("tasks", "operations", "entities", "properties", "relations", "derived_edges", "blocked_merges")}])

用时 1.5 秒；校验问题：无


,tasks,operations,entities,properties,relations,derived_edges,blocked_merges
0,468,56,5,160,8,531,6


In [7]:
ents = pd.DataFrame([{"实体": e.id, "名称": e.label, "标识字段": e.identifier, "标签字段": e.label_property,
                      "属性数": len(e.properties), "同义字段/参数": ", ".join(report["clusters"].get(e.id, []))}
                     for e in auto.objects.entities])
rels = pd.DataFrame([{"从": r.source, "到": r.target, "关系": r.label} for r in auto.objects.relations])
display(ents)
display(rels)

,实体,名称,标识字段,标签字段,属性数,同义字段/参数
0,Fund,基金,Fund.基金代码,Fund.name,87,"code, fundCode, fundCodeList, fundCodes, fundNameOrCode, identifier, productCode, 基金代码, 备选基金代码"
1,Strategy,投顾策略/组合,Strategy.策略代码,Strategy.name,63,"assetCode, assetPlanId, locatedPoCode, poCode, poCodes, prodCode, prodCodeList, productCode, strategyCode, strategyC..."
2,Account,账户,Account.assetId,Account.assetName,8,"assetId, serviceAccountId, serviceAccountIdList"
3,Index,指数,Index.indexCode,NaN,1,indexCode
4,User,用户（运行身份）,User.userId,NaN,1,userId


,从,到,关系
0,Account,Strategy,包含
1,Fund,Index,引用
2,Fund,Strategy,包含
3,Strategy,Account,引用
4,Strategy,Fund,包含
5,User,Account,拥有
6,User,Fund,可检索
7,User,Strategy,可检索


**派生边的召回率**：轨迹里每一次"上游返回的值被下游当参数用"，是否能被派生出的数据流边解释。`selection` 是上游有多个候选、从中选了的流动，`reference` 是上游只有一个值的直接引用。

In [8]:
cov = report["coverage"]
display(pd.DataFrame({k: cov[k] for k in ("all", "selection", "reference")}).T)
display(pd.DataFrame(cov["top_missed"]))

,covered,total,recall
all,3617.0,3655.0,0.990
selection,2926.0,2939.0,0.996
reference,691.0,716.0,0.965


,flow,count
0,MonteCarloSimulate → windowType → MonteCarloSimulate → frequency,16
1,GetBatchFundPerformance → stageType → BatchGetFundNavHistory → dimensionType,3
2,GetStrategyRiskInfo → strategyCode → GetFundAnnouncements → fundCodes,2
3,SearchManagerViewpoint → fundCode → SearchFunds → keyword,1
4,BatchGetFundNavHistory → type → GetAssetAllocationPlan → expectedInvestTime,1
5,BatchGetFundsDetail → largeLimitAmount → AnalyzeAssetLiability → annualExpense,1
6,BatchGetStrategiesComposition → 策略代码 → AnalyzeFundRisk → fundCodes,1
7,AnalyzeCashFlow → analysisData → AnalyzeInvestmentPerformance → totalInvestableAssets,1
8,BatchGetFundNavHistory → type → BatchGetFundNavHistory → dimensionType,1
9,GetBatchFundPerformance → stageType → GetAssetAllocationPlan → expectedInvestTime,1


## 4. 第 6 步：人工整理（示范）

自动建图把 `assetPlanId`（资产配置方案编号）并进了 Strategy，因为它和策略代码共享了一些上下游。下面做几处典型的人工整理：拆出 AssetPlan 实体、补实体含义和规则、指定更合适的标签字段。整理后再校验一次。

In [9]:
cur = auto.model_copy(deep=True)
cur.id, cur.label = "finmcp-curated", "FinMCP-Bench（自动建图 + 人工整理）"

# 1) 拆出资产配置方案
cur.objects.entities.append(Entity(id="AssetPlan", label="资产配置方案", identifier="AssetPlan.assetPlanId",
                                   description="按投资期限、目标收益和回撤生成的资产配置方案",
                                   properties=[Property(id="AssetPlan.assetPlanId", name="assetPlanId", description="方案编号")]))
strategy_id = cur.entity("Strategy").properties[0]
strategy_id.aliases = [a for a in strategy_id.aliases if a != "assetPlanId"]
moved = 0
for o in cur.operations.operations:
    for p in o.inputs:
        if p.name == "assetPlanId":
            p.property, moved = "AssetPlan.assetPlanId", moved + 1
    for b in o.outputs:
        if b.path.rsplit(".", 1)[-1] == "assetPlanId":
            b.property, moved = "AssetPlan.assetPlanId", moved + 1
cur.objects.relations.append(Relation(id="r_plan", source="User", target="AssetPlan", label="可生成"))

# 2) 含义和规则
cur.entity("Fund").description = "公募基金，标识是 6 位基金代码"
cur.entity("Fund").rules = ["基金代码和策略代码是两套编号，不能混用"]
cur.entity("Strategy").description = "投顾策略/组合，标识形如 ZH000095 或 LONG_WIN"
cur.entity("Strategy").rules = ["策略的历史收益不代表未来表现"]

# 3) 标签字段：优先用中文名称字段
for eid, names in (("Strategy", ["策略名称", "strategyName"]), ("Fund", ["基金名称", "基金简称", "fundName"])):
    found = next((n for n in names if cur.property(f"{eid}.{n}")), None)
    if found:
        cur.entity(eid).label_property = f"{eid}.{found}"
print("标签字段：", {e.id: e.label_property for e in cur.objects.entities})

print(f"改动了 {moved} 处绑定；校验：", S.validate(cur) or "通过")
display(pd.DataFrame({k: build.coverage(cur, report["flows"])[k] for k in ("all", "selection", "reference")}).T)

标签字段： {'Fund': 'Fund.基金简称', 'Strategy': 'Strategy.策略名称', 'Account': 'Account.assetName', 'Index': None, 'User': None, 'AssetPlan': None}
改动了 3 处绑定；校验： 通过


,covered,total,recall
all,3611.0,3655.0,0.988
selection,2926.0,2939.0,0.996
reference,685.0,716.0,0.957


可选：让大模型给自动归纳出的属性写含义（需要在 `keys.py` 里填好大模型的 key；会产生少量费用）。默认关闭。

In [10]:
RUN_LLM_CURATION = False
if RUN_LLM_CURATION:
    from jif.llm import LLM
    keys = config.load_keys()
    llm = LLM(keys["LLM_BASE_URL"], keys["LLM_API_KEY"], keys["LLM_MODEL"])
    for e in cur.objects.entities:
        names = [p.name for p in e.properties if not p.description][:40]
        if not names:
            continue
        out = llm.chat([{"role": "system", "content": "你是金融数据字典编辑。只输出 JSON。"},
                        {"role": "user", "content": f"实体 {e.id}（{e.label}）有这些字段：{names}。请为每个字段写一句中文含义，输出 {{字段名: 含义}}。"}],
                       tag="curation")
        meanings = json.loads(out["message"]["content"][out["message"]["content"].find("{"):out["message"]["content"].rfind("}") + 1])
        for p in e.properties:
            p.description = p.description or meanings.get(p.name, "")
    print("已补充属性含义")

## 5. 披露深度与接口桩大小

agent 看到的是**披露范围内的操作**渲染出的接口桩。下表对比不同披露深度下的操作数和接口桩大小；作为参照，也给出 ReAct 每一轮都要发送的工具定义大小（从日志反推的 56 个工具的 JSON Schema）。

FinMCP 的工具大多是"可检索"的（不依赖用户身份），所以从用户出发 1 跳就能看到大部分操作；对象范围披露在工具更多、更分散的场景里才更明显（见实验设计 A3/A4）。

In [11]:
from jif.replay import ReplayStore
react_chars = len(json.dumps([{"type": "function", "function": {"name": s["name"], "description": s["description"],
                                                                "parameters": s["input_schema"]}}
                              for s in ReplayStore().tool_specs()], ensure_ascii=False))
rows = []
for h in (0, 1, 2, None):
    d = S.disclose(cur, h) if h is not None else {"entities": [e.id for e in cur.objects.entities],
                                                    "operations": [o.id for o in cur.operations.operations]}
    rows.append({"披露深度": "全部" if h is None else h, "实体": len(d["entities"]), "操作": len(d["operations"]),
                 "SOG 接口桩字符数": len(S.render_stubs(cur, d["operations"])),
                 "平铺接口桩字符数": len(S.render_stubs(cur, d["operations"], mode="flat"))})
display(pd.DataFrame(rows))
print("参照：ReAct 每轮发送的工具定义约", react_chars, "字符（每一轮都要重复发送）")

,披露深度,实体,操作,SOG 接口桩字符数,平铺接口桩字符数
0,0,1,24,14769,4585
1,1,5,55,44498,8928
2,2,6,56,47138,9007
3,全部,6,56,47138,9007


参照：ReAct 每轮发送的工具定义约 26097 字符（每一轮都要重复发送）


In [12]:
print(S.render_stubs(cur, ["StrategySearchByKeyword", "GetUserAssetHoldingDistribution"]))

# ---- 实体（对象图） ----
# Fund（基金）：标识字段 基金代码，名称字段 基金简称。公募基金，标识是 6 位基金代码
#   规则：基金代码和策略代码是两套编号，不能混用
# Strategy（投顾策略/组合）：标识字段 策略代码，名称字段 策略名称。投顾策略/组合，标识形如 ZH000095 或 LONG_WIN
#   规则：策略的历史收益不代表未来表现

# ---- 操作（操作图）：只能用关键字参数调用 ----
def StrategySearchByKeyword(keyword: str, pageNum: int = None, pageSize: int = None, category: str = None, size: int = None) -> StrategySearchByKeyword_R:
    """StrategySearchByKeyword：返回字段 pageSize, rows, status, hasMoreRows"""
    class StrategySearchByKeyword_R:
        pageSize: int
        rows: list[StrategySearchByKeyword_R_rows]
        status: str
        hasMoreRows: bool
    class StrategySearchByKeyword_R_rows:  # Strategy 记录
        策略代码: str
        策略名称: str
        策略风险等级: str
        策略简介: str
        策略描述: str
        策略成立时间: str

def GetUserAssetHoldingDistribution(fundCodes: list = None, poCodes: list = None) -> GetUserAssetHoldingDistribution_R:
    """GetUserAssetHoldingDistribution：返回字段 fundAssetDistributions, poAssetDistributions（userId 由运行时注入

## 6. 保存

In [13]:
auto.save(config.ARTIFACTS / "sog_finmcp_auto.json")
cur.save(config.ARTIFACTS / "sog_finmcp.json")
print("已保存：", config.ARTIFACTS / "sog_finmcp_auto.json", "和", config.ARTIFACTS / "sog_finmcp.json")

已保存： /Users/nelson/Documents/JevAsIF/Notebook/artifacts/sog_finmcp_auto.json 和 /Users/nelson/Documents/JevAsIF/Notebook/artifacts/sog_finmcp.json
